# Breeze ASR 25 — 晶晶體 arm(X-breeze)

**這個 notebook 在測什麼:** [Breeze ASR 25](https://huggingface.co/MediaTek-Research/Breeze-ASR-25)
(MediaTek Research,Whisper-large-v2 微調,針對台灣華語與中英夾雜優化,Apache 2.0)
在 kikemu exp2 的**同一批語料、同一批聲學條件**上,英文術語召回率能不能贏
Speechmatics。它沒有 keyterm 機制,所以代表的是「純靠模型本身」能走多遠。

**你要做的事:** 執行階段 → 全部執行。第一次會要求授權 Google 雲端硬碟。
音檔請先放進雲端硬碟(見下一格說明),其餘全自動。

---

### 開始前只有一件事要準備

把 `S1.wav` `S2.wav` `S3.wav` 放進雲端硬碟的 **`kikemu-corpus/`** 資料夾
(沒有就自己開一個)。這三個檔是 exp2 的原始切片,依授權不放進 repo,
所以要由你提供;放好之後每次重跑都會自動沿用。

> Phase A 只需要 `S1.wav` 與 `S3.wav`;Phase B 三個都要。

放好後就按「執行階段 → 全部執行」,不用改任何程式碼。

## 1. 環境與 GPU 檢查

沒有 GPU 就直接報錯——large-v2 在 CPU 上一個檔要跑幾十分鐘,不要靜默降級。

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "(nvidia-smi 沒有輸出)")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "沒有 GPU。請到「執行階段 → 變更執行階段類型 → 硬體加速器 → T4 GPU」再重跑。\n"
        "(Whisper-large-v2 在 CPU 上跑 300 秒音檔要數十分鐘,評測不該這樣跑。)")
print("GPU OK:", torch.cuda.get_device_name(0))

%pip install -q "transformers>=4.44" librosa soundfile
import transformers, librosa
print("transformers", transformers.__version__, "| torch", torch.__version__)

## 2. 參數(要改就改這裡)

In [ ]:
REPO      = "clarencechien/kikemu"
BRANCH    = "claude/improve-experiment-credibility-c8ekb2"
PHASE     = "A"          # "A" = 三段快篩(S1-M0 / S1-M3 / S3-M0);"B" = 3 段 × 4 條件 = 12 檔
MODEL_ID  = "MediaTek-Research/Breeze-ASR-25"
REVISION  = "cffe7ccb404d025296a00758d0a33468bec3a9d0"  # 釘住 commit;要跑最新改成 "main"
DTYPE     = "float16"    # 評測用,不要換 int8(會混入量化損失這個變因)
LANG_ARGS = [None, "zh"] # 兩種都跑:中英夾雜下語言標記會直接影響英文術語會不會被中文化
DRIVE_DIR = "kikemu-corpus"   # 雲端硬碟裡放 S1/S2/S3.wav 的資料夾
USE_DRIVE = True         # 關掉的話音檔要自己放到 /content/kikemu/exp2/corpus/wav/
PUSH_BACK = False        # True 需要 GitHub PAT;預設關,結果用下載的

## 3. 取得 repo、音檔與聲學條件

三件事,全部自動:

1. `git clone` repo(淺層)
2. 從雲端硬碟拿 `S1/S2/S3.wav`(原始切片)
3. 下載噪音素材(MIT IR Survey + DEMAND,公開來源)→ 跑 repo 裡**原本那支**
   `exp2/scripts/degrade.py` 生出 M0–M3

最後對照 `exp2/corpus/audio_manifest.json` 的 SHA256。**指紋相符 = 你手上的檔案
與 X / G 各 arm 當初實際跑的是同一份**,結果才可以直接放進同一張表比較。
(`degrade.py` 已驗證為位元決定性:同樣的輸入重跑,12 個檔的 sha256 全部相同。)

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

WORK = Path("/content/kikemu")
if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://github.com/{REPO}.git", str(WORK)], check=True)
os.chdir(WORK)
MANIFEST = json.loads((WORK / "exp2/corpus/audio_manifest.json").read_text())

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# --- 3a. 原始切片:雲端硬碟 → exp2/corpus/wav/
# NEED = 這個 Phase「非有不可」的;有放更多就一起搬進來,不要只挑 NEED——
# degrade.py 是照 S1/S2/S3 整批跑的,少搬一個它就少產一段,之後改跑 Phase B
# 還得再回來重做一次。
ALL_SEGS = ["S1", "S2", "S3"]
NEED_SEGS = ["S1", "S3"] if PHASE == "A" else ALL_SEGS
WAVDIR = WORK / "exp2/corpus/wav"
WAVDIR.mkdir(parents=True, exist_ok=True)

drive_root = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive") / DRIVE_DIR
    drive_root.mkdir(parents=True, exist_ok=True)

have = []
for seg in ALL_SEGS:
    dst = WAVDIR / f"{seg}.wav"
    if not dst.exists():
        src = (drive_root / f"{seg}.wav") if drive_root else None
        if src and src.exists():
            shutil.copy(src, dst)
    if dst.exists():
        have.append(seg)
print("找到的原始切片:", ", ".join(have) or "(無)")

missing = [s for s in NEED_SEGS if s not in have]
if missing:
    raise RuntimeError(
        f"缺少原始切片:{', '.join(f'{s}.wav' for s in missing)}\n"
        f"請把它們放進雲端硬碟的 {DRIVE_DIR}/ 資料夾後,重新執行這一格。\n"
        "(音檔依授權不隨 repo 散布,所以要由你提供;放一次之後就會一直沿用。)")

for seg in have:
    want = MANIFEST["source_wav"][seg]["sha256"]
    got = sha256(WAVDIR / f"{seg}.wav")
    print(f"  {seg}.wav  {'✅ 與各 arm 同源' if got == want else '⚠️ 指紋不符(來源不同,結果不可直接與其他 arm 併表)'}")

# --- 3b. 噪音素材(公開來源,大小與 sha256 都對過)
NSRC = WORK / "corpus/noise_src"
NSRC.mkdir(parents=True, exist_ok=True)
for name, info in MANIFEST["noise_src"].items():
    if info["used_by"] != "exp2":
        continue                      # PCAFETER 是 exp1 用的,這裡不需要
    dst = NSRC / name
    cache = (drive_root / name) if drive_root else None
    if not dst.exists() and cache and cache.exists():
        shutil.copy(cache, dst)
    if not dst.exists():
        print(f"  下載 {name}({info['bytes']/1e6:.0f}MB)…")
        urllib.request.urlretrieve(info["url"], dst)
        if cache:
            shutil.copy(dst, cache)   # 存回雲端硬碟,下次不用再抓
    ok = sha256(dst) == info["sha256"]
    print(f"  {name}  {'✅' if ok else '❌ sha256 不符,請刪掉重抓'}")
    if not ok:
        raise RuntimeError(f"{name} 下載損毀,請刪掉重跑這一格")

# --- 3c. 用 repo 原本那支 degrade.py 生 M0–M3(不要在這裡重寫一套)
# 不要用 check=True 就算了:子行程死掉時只會看到 exit code,查不出所以然。
r = subprocess.run([sys.executable, "exp2/scripts/degrade.py"],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(r.stderr.strip())
    raise RuntimeError("degrade.py 失敗,原因見上方 stderr")

PHASE_A_FILES = ["S1__M0", "S1__M3", "S3__M0"]
targets = PHASE_A_FILES if PHASE == "A" else sorted(MANIFEST["conditions"])
print("\n聲學條件指紋:")
for stem in targets:
    p = WORK / "exp2/corpus/conditions" / f"{stem}.wav"
    if not p.exists():
        raise RuntimeError(f"{stem}.wav 沒有產出——通常是缺對應的原始切片 "
                           f"({stem.split('__')[0]}.wav),請放進雲端硬碟再重跑這一格")
    ok = sha256(p) == MANIFEST["conditions"][stem]["sha256"]
    print(f"  {stem}  {'✅ bit-identical' if ok else '⚠️ 不符'}")
print(f"\n本次要跑 {len(targets)} 個檔 × {len(LANG_ARGS)} 種語言設定 = {len(targets)*len(LANG_ARGS)} 次推論")

## 4. 載入模型

依模型卡的建議建構 pipeline:**`chunk_length_s=0`(sequential)**。
chunked 模式比較快,但那樣量到的是分塊策略的差異,不是模型本身。

In [ ]:
import time, torch
from transformers import (AutomaticSpeechRecognitionPipeline, WhisperForConditionalGeneration,
                          WhisperProcessor)

t0 = time.time()
processor = WhisperProcessor.from_pretrained(MODEL_ID, revision=REVISION)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=REVISION, torch_dtype=torch.float16).to("cuda").eval()
asr = AutomaticSpeechRecognitionPipeline(
    model=model, tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=0,          # sequential long-form(模型卡建議)
    torch_dtype=torch.float16, device="cuda",
)
assert asr._preprocess_params.get("chunk_length_s") in (0, None), "chunk_length_s 沒有生效"
print(f"模型載入完成 {time.time()-t0:.0f}s | revision={REVISION[:12]} | dtype=float16 | chunk_length_s=0")

## 5. 推論 + 寫出結果

- 兩種語言設定各跑一次:**不指定**(`Xbrz_auto`)與 **`language="zh"`**(`Xbrz_zh`)
- 輸出欄位對齊既有 arm(`exp2/scripts/score.py` 讀的是 **`transcript`**),
  所以 `score.py` 不用改就吃得下
- 已經有結果的檔會跳過,斷線重連可以直接接著跑

In [ ]:
import json, time, librosa
from pathlib import Path

RAW = WORK / "exp2/results/raw"
meta_env = {
    "model": MODEL_ID, "revision": REVISION, "dtype": DTYPE,
    "chunk_length_s": 0, "return_timestamps": True,
    "transformers": transformers.__version__, "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0), "phase": PHASE,
}

for lang in LANG_ARGS:
    arm = "Xbrz_zh" if lang else "Xbrz_auto"
    out = RAW / arm
    out.mkdir(parents=True, exist_ok=True)
    (out / "_meta.json").write_text(json.dumps({**meta_env, "language_arg": lang},
                                               ensure_ascii=False, indent=1))
    for stem in targets:
        dst = out / f"{stem}.json"
        if dst.exists():
            print(f"  跳過(已有){arm}/{stem}")
            continue
        wav = WORK / "exp2/corpus/conditions" / f"{stem}.wav"
        audio, _ = librosa.load(wav, sr=16000, mono=True)
        # 把音檔指紋寫進結果:日後光看這個 JSON 就知道它跑的是不是與其他 arm 同一份
        wav_sha = sha256(wav)
        kw = {"return_timestamps": True}
        if lang:
            kw["generate_kwargs"] = {"language": lang, "task": "transcribe"}
        t0 = time.time()
        r = asr(audio.copy(), **kw)
        elapsed = time.time() - t0
        seg, cond = stem.split("__")
        dst.write_text(json.dumps({
            "arm": arm,
            "file": f"{stem}.wav",
            "audio_s": round(len(audio) / 16000, 1),
            "transcript": r["text"].strip(),          # ← score.py 讀這一欄
            "segments": [{"start": c["timestamp"][0], "end": c["timestamp"][1],
                          "text": c["text"]} for c in r.get("chunks", [])],
            "meta": {**meta_env, "language_arg": lang, "elapsed_sec": round(elapsed, 1),
                     "audio_sha256": wav_sha,
                     "audio_matches_manifest": wav_sha == MANIFEST["conditions"][stem]["sha256"]},
        }, ensure_ascii=False, indent=1))
        print(f"  {arm}/{stem}  {elapsed:.0f}s  {len(r['text'])} 字")
print("\n完成")

# --- language 參數的健全性檢查 ---------------------------------------
# 若 Xbrz_auto 與 Xbrz_zh 輸出完全相同,有兩種可能:(a) 自動偵測本來就判成中文,
# (b) generate_kwargs 根本沒被吃進去。拿一個檔跑 language="en",輸出若變了
# 就排除 (b)。只多跑一次,很便宜,但少了它「兩種設定沒差」這句話沒有意義。
if len(LANG_ARGS) > 1 and targets:
    stem = targets[0]
    a = json.loads((RAW / "Xbrz_auto" / f"{stem}.json").read_text())["transcript"]
    z = json.loads((RAW / "Xbrz_zh" / f"{stem}.json").read_text())["transcript"]
    if a == z:
        wav = WORK / "exp2/corpus/conditions" / f"{stem}.wav"
        audio, _ = librosa.load(wav, sr=16000, mono=True)
        en = asr(audio.copy(), return_timestamps=True,
                 generate_kwargs={"language": "en", "task": "transcribe"})["text"].strip()
        (RAW / "_lang_sanity.json").write_text(json.dumps(
            {"stem": stem, "auto_eq_zh": True, "en_differs": en != a,
             "en_head": en[:200]}, ensure_ascii=False, indent=1))
        print(f"language 健全性:auto == zh;改成 en 後輸出"
              f"{'有變化 → 參數確實生效' if en != a else '仍然相同 → 參數可能沒被吃進去,存疑'}")

## 6. Phase A 判讀

快篩要回答的只有一件事:**M3(混響 + 12dB 多人交談底噪)會不會崩。**
Breeze 的中文訓練資料全部是合成語音,真實聲學條件下的穩健性是最大未知數。

崩掉的樣子:輸出歸零、整段重複同一句、或英文術語全滅。
**M3 崩了就不必做 Phase B**,在報告記一筆「合成訓練資料在真實噪音下不穩」即可。

In [ ]:
from IPython.display import Markdown, display

terms = json.loads((WORK / "exp2/corpus/terms.json").read_text())

lines = []
for stem in targets:
    seg = stem.split("__")[0]
    want = [t["term"] for t in terms[seg]]
    lines.append(f"### {stem}")
    for lang in LANG_ARGS:
        arm = "Xbrz_zh" if lang else "Xbrz_auto"
        f = RAW / arm / f"{stem}.json"
        if not f.exists():
            continue
        d = json.loads(f.read_text())
        tx = d["transcript"]
        hit = [t for t in want if t.lower() in tx.lower()]
        lines.append(f"**{arm}**(字面命中 {len(hit)}/{len(want)} 個術語,"
                     f"{d['meta']['elapsed_sec']}s):{', '.join(hit[:12]) or '—'}")
        lines.append(f"> {tx[:600]}{'…' if len(tx) > 600 else ''}")
    lines.append("")
display(Markdown("\n\n".join(lines)))
print("字面命中只是給眼睛看的快照,正式數字要跑 exp2/scripts/score.py(含變體與失敗模式分類)")

## 7. 把結果帶走

預設下載一個 zip。要直接回寫 repo 就把 `PUSH_BACK` 設 True 並填 PAT
(需要 `repo` 權限;**貼在這裡的 token 會留在 notebook 輸出裡,用完請撤銷**)。

In [ ]:
import shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d-%H%M")
zip_base = f"/content/x-breeze-{PHASE}-{stamp}"
shutil.make_archive(zip_base, "zip", RAW, ".")
print("已打包:", zip_base + ".zip")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("自動下載失敗,請從左側檔案面板手動下載:", e)

if PUSH_BACK:
    import getpass
    pat = getpass.getpass("GitHub PAT: ")
    subprocess.run(["git", "config", "user.email", "colab@kikemu"], cwd=WORK, check=True)
    subprocess.run(["git", "config", "user.name", "kikemu colab"], cwd=WORK, check=True)
    subprocess.run(["git", "add", "exp2/results/raw/Xbrz_auto", "exp2/results/raw/Xbrz_zh"],
                   cwd=WORK, check=True)
    subprocess.run(["git", "commit", "-m", f"exp2: X-breeze arm (phase {PHASE})"], cwd=WORK, check=True)
    subprocess.run(["git", "push", f"https://{pat}@github.com/{REPO}.git", f"HEAD:{BRANCH}"],
                   cwd=WORK, check=True)
    print("已推回", BRANCH)

---

## ⚠️ 這個 arm 的偏誤警告(報告裡不可省略)

**Breeze ASR 25 的致謝名單包含李宏毅教授,而本評測的語料正是他的課程。**

中文訓練資料雖為合成,但團隊對 ML 課程的語域與術語分佈極為熟悉,
**X-breeze 在這批語料上的表現很可能偏高**。

所以:

1. 報告中要明確標註此潛在偏誤
2. Phase B 之前要補一段**領域外**的中英夾雜對照語料(真實科技業會議或 Podcast,
   語域非 ML),看優勢是否仍在
3. 若只在 ML 語料上領先,結論只能寫「**領域內**表現優異」,
   不能寫成「通用台灣華語引擎更好」